# Model H (Telugu) — Kaggle Pretraining Notebook

In [13]:
import os
from pathlib import Path

def find_bundle_root(base="/kaggle/input", need=("TELUGU", "src")):
    for root, dirs, files in os.walk(base):
        if all(n in dirs for n in need):
            return Path(root)
    raise FileNotFoundError(
        f"Couldn't find a folder containing {need} under {base}. "
        "Run `!find /kaggle/input -maxdepth 3` to inspect what got mounted, "
        "and confirm the dataset is attached."
    )

INPUT_ROOT = find_bundle_root()
print("Bundle found at:", INPUT_ROOT)

Bundle found at: /kaggle/input/datasets/ervarishitha11/lma-telugu-phase2


## Build a writable working copy
Symlinks the read-only tokenizer/data (and `src/`, `tests/`) straight out of the mounted dataset instead of copying several GB, and creates a real writable `outputs/` dir for checkpoints/cache/report.

In [14]:
import shutil

WORK = Path("/kaggle/working/phase2")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

os.symlink(INPUT_ROOT / "src", WORK / "src")
os.symlink(INPUT_ROOT / "tests", WORK / "tests")

(WORK / "TELUGU").mkdir()
os.symlink(INPUT_ROOT / "TELUGU" / "PHASE1", WORK / "TELUGU" / "PHASE1")  # tokenizer + data splits, read-only, fine
(WORK / "TELUGU" / "PHASE2").mkdir()
os.symlink(INPUT_ROOT / "TELUGU" / "PHASE2" / "configs", WORK / "TELUGU" / "PHASE2" / "configs")
for s in ["train_telugu.py", "eval_telugu.py"]:
    shutil.copy(INPUT_ROOT / "TELUGU" / "PHASE2" / s, WORK / "TELUGU" / "PHASE2" / s)
(WORK / "TELUGU" / "PHASE2" / "outputs").mkdir(parents=True)  # writable: checkpoints/cache/report

print("Ready:")
os.system(f"find {WORK} -maxdepth 3 | sort")

Ready:
/kaggle/working/phase2
/kaggle/working/phase2/src
/kaggle/working/phase2/TELUGU
/kaggle/working/phase2/TELUGU/PHASE1
/kaggle/working/phase2/TELUGU/PHASE2
/kaggle/working/phase2/TELUGU/PHASE2/configs
/kaggle/working/phase2/TELUGU/PHASE2/eval_telugu.py
/kaggle/working/phase2/TELUGU/PHASE2/outputs
/kaggle/working/phase2/TELUGU/PHASE2/train_telugu.py
/kaggle/working/phase2/tests


sh: 0: getcwd() failed: No such file or directory


0

## Install dependencies + connect Google Drive (rclone)

In [15]:
os.system("pip install -q sentencepiece sacrebleu rouge-score")
os.system("curl https://rclone.org/install.sh | sudo bash")

from kaggle_secrets import UserSecretsClient
conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(conf_dir, exist_ok=True)

raw_secret = UserSecretsClient().get_secret("RCLONE_CONF")
# Kaggle's Secret value field flattens real newlines into spaces, so the config must be
# stored with newlines encoded as literal \n -- decode that back into real line breaks
# before writing, or rclone can't find the [gdrive] section.
conf_content = raw_secret.replace("\\n", "\n")
with open(f"{conf_dir}/rclone.conf", "w") as f:
    f.write(conf_content)

os.system("rclone lsd gdrive:")  # sanity check -- should list your Drive's top-level folders

sh: 0: getcwd() failed: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
sh: 0: getcwd() failed: No such file or directory
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
100  4734  100  4734    0     0   9463      0 --:--:-- --:--:-- --:--:--  9468
job-working-directory: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
chdir: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory



The latest version of rclone rclone v1.75.0 is already installed.



sh: 0: getcwd() failed: No such file or directory
2026/09/03 14:18:40 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


           0 2026-08-21 05:27:42        -1 2025201027_LMA_Phase1
           0 2026-08-16 08:50:01        -1 Colab Notebooks
           0 2026-09-02 14:24:00        -1 LMA_Project1


0

## Resume safety: pull down any checkpoint from a previous session
(No-op the very first time.)

In [16]:
os.system(f"rclone copy gdrive:LMA_Project1/TELUGU_checkpoints {WORK}/TELUGU/PHASE2/outputs/checkpoints -P")

sh: 0: getcwd() failed: No such file or directory
2026/09/03 14:18:40 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 1.156 GiB, 0%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 11
Transferred:            0 / 10, 0%
Elapsed time:         0.5s
Transferring:
 *                           checkpoint_final.pt:  0% / 295.880 MiB, 0 B/s, -
 *                          checkpoint_latest.pt:  0% / 295.880 MiB, 0 B/s, -
 *                      checkpoint_step_25000.pt:  0% / 295.881 MiB, 0 B/s, -
 *                      checkpoint_step_30000.pt:  0% / 295.881 MiB, 0 B/s, -Transferred:   	  160.500 MiB / 1.156 GiB, 14%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 11
Transferred:            0 / 10, 0%
Elapsed time:         1.0s
Transferring:
 *                           checkpoint_final.pt: 12% / 295.880 MiB, 0 B/s, -
 *                          checkpoint_latest.pt: 21% / 295.880 MiB, 0 B/s, -
 *                      checkpoint_step_25000.pt:  7% / 295.881 MiB, 0 B/s, -
 *                      checkpoint_step_30000.pt: 13% / 295.881 MiB, 0 B/s, -Transferred:   	 

0

## Start the background Drive-sync loop
Syncs every 5 minutes for the life of the session, so at most ~5 minutes of work is ever at risk if the session dies without warning.

In [17]:
sync_sh = f"""
while true; do
  rclone sync {WORK}/TELUGU/PHASE2/outputs/checkpoints gdrive:LMA_Project1/TELUGU_checkpoints --include \"checkpoint_latest.pt\" --include \"checkpoint_final.pt\" --include \"checkpoint_step_*.pt\" --include \"training_history.json\"
  sleep 300
done
"""
open("/kaggle/working/sync_telugu.sh", "w").write(sync_sh)
os.system("nohup bash /kaggle/working/sync_telugu.sh > /kaggle/working/sync_telugu.log 2>&1 &")
print("Background Drive sync started.")

Background Drive sync started.


sh: 0: getcwd() failed: No such file or directory


## Sanity check before spending GPU hours
Confirms the GPU is visible and the model/tokenizer/checkpoint pipeline loads correctly against the working bundle.

In [18]:
import torch
print("CUDA available:", torch.cuda.is_available(), "-",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

os.chdir(str(WORK))

import sys
sys.path.insert(0, str(WORK))
from src.model import GPTLanguageModel
from src.dataset import SentencePieceTokenizer

model = GPTLanguageModel(vocab_size=12000, d_model=512, n_layer=6, n_head=8, d_ff=2048,
                          max_seq_len=512, dropout=0.1, tie_weights=True)
params = model.count_parameters()
print("Total trainable parameters:", f"{params['total_trainable']:,}")
assert 24_000_000 < params['total_trainable'] < 26_000_000
assert model.verify_causal_masking(), "causal mask check failed"
print("Causal masking check: PASSED")

tok = SentencePieceTokenizer(str(WORK / "TELUGU/PHASE1/outputs/tokenizer/telugu_bpe.model"))
print("Tokenizer vocab size:", tok.vocab_size)
print("Sample roundtrip:", tok.decode(tok.encode("తెలుగు సమాచారం")))
print("\nALL SANITY CHECKS PASSED")

CUDA available: True - Tesla T4
Total trainable parameters: 25,321,472
Causal masking check: PASSED
Tokenizer vocab size: 12000
Sample roundtrip: తెలుగు సమాచారం

ALL SANITY CHECKS PASSED


## Pretrain Model H (Telugu)
Auto-resumes from `checkpoint_latest.pt` if one was pulled down in the resume-safety cell above.

In [ ]:
%cd {WORK}
!python TELUGU/PHASE2/train_telugu.py

## Evaluate the trained checkpoint, then push everything to Drive
Run once `checkpoint_final.pt` exists (training reached `max_steps`), or any time for an interim eval against `checkpoint_latest.pt`.

In [19]:
!python TELUGU/PHASE2/eval_telugu.py

os.system(f"rclone sync {WORK}/TELUGU/PHASE2/outputs/checkpoints gdrive:LMA_Project1/TELUGU_checkpoints -P")
os.system(f"rclone copy {WORK}/TELUGU/PHASE2/outputs/report gdrive:LMA_Project1/TELUGU_checkpoints/report -P")
print("Done -- Telugu checkpoints, logs, and eval report are all on Drive.")

=== Evaluating Model H (Telugu) on cuda ===
Loading checkpoint from /kaggle/working/phase2/TELUGU/PHASE2/outputs/checkpoints/checkpoint_final.pt
[TextDataset] Cached 4,918 docs / 2,810,564 tokens -> val.v12000.dall.ids.bin (11.2 MB)

--- Intrinsic LM Metrics ---
Val Loss:   3.6565
Perplexity: 38.7273
BPB:        0.4826

--- Extracted 10 Prefix/Reference Pairs from test.jsonl ---

Prompt: Infertility: మహిళల్లో సంతానోత్పత్తి సమస్య (In
  [Temp 0.5]: Infertility: మహిళల్లో సంతానోత్పత్తి సమస్య (Infertility) ఏర్పడిందని నిపుణులు చెబుతున్నారు. గర్భధారణ సమయంలో
  [Temp 1.0]: Infertility: మహిళల్లో సంతానోత్పత్తి సమస్య (Infertility: గర్భం విషయంలో ఇది చాలా ముఖ్యమైనది.
  [Temp 1.5]: Infertility: మహిళల్లో సంతానోత్పత్తి సమస్య (Indi జీవనశైలి చేసుకోవాలని ఐట్యూబ్ దిగిన పరిశోధ జర్నల్స్లో పరిశోధ

Prompt: విశాఖపట్నం, ఫిబ్రవరి 20: పేదలందరికీ ప్రభుత్వం ఇళ్లు అందించేందుకు చేపడుతున్న భూసే
  [Temp 0.5]: విశాఖపట్నం, ఫిబ్రవరి 20: పేదలందరికీ ప్రభుత్వం ఇళ్లు అందించేందుకు చేపడుతున్న భూసేకరణకు వ్యతిరేకంగా రాష్ట్ర ప్రభుత

2026/09/03 14:20:04 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                10 / 10, 100%, Listed 22
Elapsed time:         0.4s


2026/09/03 14:20:56 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 0 / 4, 0%, Listed 9
Elapsed time:         0.5s
Checking:
 *                         attn_layer1_head1.png: checking
 *                attn_layer1_multihead_grid.png: checking
 *                         attn_layer6_head1.png: checking
 *                attn_layer6_multihead_grid.png: checkingTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 4 / 4, 100%, Listed 9
Elapsed time:         0.9s
Done -- Telugu checkpoints, logs, and eval report are all on Drive.


## Generate report figures and push to Drive

In [21]:
!python src/generate_report_visuals.py

[Visuals] Architecture diagram saved -> report/figures/architecture_diagram.png
[Visuals] Corpus comparison chart saved -> report/figures/corpus_comparison.png
[Visuals] Convergence curve saved -> report/figures/pretrain_loss_curve.png
